# Aquatic Inpainting - Colab (todo automatico)
**Run all.** Hace una prueba de humo automatica antes del entrenamiento largo y aborta si algo falla.
Credenciales (en este orden): Colab Secrets -> archivo `.env` en Drive (`PERSIST/.env`, con lineas `HF_TOKEN=...`, `KAGGLE_USERNAME=...`, `KAGGLE_KEY=...`) -> te las pide aqui (campo oculto).
Necesitas runtime con GPU y la cuenta de Google que tiene Colab Pro.

In [ ]:
REPO    = 'Vasco2510/lab3-aquatic'
PERSIST = '/content/drive/MyDrive/UTEC/DEEP/lab3_aquatic'
PRESET  = 'night'     # smoke | full | night (3 semillas, tope 8h)
EXTRA   = '--submit'  # '' para no enviar a Kaggle. Ej: '--epochs 25 --hours 6 --seeds 2'
import os, subprocess, getpass
from google.colab import userdata, drive
try: drive.mount('/content/drive')
except Exception as e: print('!! SIN DRIVE (no se guardaran checkpoints):', e); PERSIST = ''
def load_env_file(path):
    if path and os.path.exists(path):
        for line in open(path, encoding='utf-8'):
            if '=' in line and not line.lstrip().startswith('#'):
                k, v = line.strip().split('=', 1); os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
for k in ['HF_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY']:
    try: os.environ[k] = userdata.get(k)
    except Exception: pass
load_env_file(os.path.join(PERSIST, '.env') if PERSIST else '')
if not os.environ.get('HF_TOKEN'): os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
if '--submit' in EXTRA and not (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')):
    print('!! Faltan KAGGLE_USERNAME/KAGGLE_KEY: se entrenara pero NO se enviara a Kaggle'); EXTRA = EXTRA.replace('--submit', '')
r = subprocess.run('nvidia-smi -L', shell=True, capture_output=True, text=True)
print(r.stdout.strip() or '!! SIN GPU: cambia el runtime a GPU (cuenta con Colab Pro) antes de seguir')

In [ ]:
os.chdir('/content')  # idempotente: re-ejecutar esta celda no anida carpetas
if os.path.isdir('/content/repo/.git'): subprocess.check_call(['git', '-C', '/content/repo', 'pull', '-q'])
else: subprocess.check_call(['git', 'clone', '-q', f'https://github.com/{REPO}.git', '/content/repo'])
os.chdir('/content/repo')
!pip -q install -r requirements.txt

In [ ]:
try: PRESET, EXTRA, PERSIST
except NameError: raise SystemExit('Faltan las variables: ejecuta primero la celda 1 (la de configuracion).')
import sys
cmd = [sys.executable, '-u', 'src/pipeline.py', '--preset', PRESET] + (['--persist', PERSIST] if PERSIST else []) + EXTRA.split()
print('$', ' '.join(cmd))
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd='/content/repo')
for line in p.stdout:
    if 'UserWarning' in line or 'torch.from_numpy' in line: continue   # ruido
    print(line, end='')
p.wait()
if p.returncode: raise SystemExit(f'el pipeline termino con error (codigo {p.returncode})')